# E03 — Home Credit Bureau feature-family ablation

This research notebook keeps the locked E01 application-only baseline and adds
36 candidates derived only from `bureau.csv` and `bureau_balance.csv`.
Dataset-specific loading, aggregation and joins live here; reusable numeric,
CV, model and artifact utilities are imported from `src/`. The committed
default is `screening`, enabled only after Checkpoint 3 approval.

In [ ]:
# 1. Clone the locked public source and import reusable utilities
import gc
import importlib
import platform
import subprocess
import sys
import time
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from pandas.testing import assert_frame_equal

REPO_URL = "https://github.com/ManhTanTran/Qaci-datascience.git"
REPO_BRANCH = "main"
REPO_COMMIT = "8ea148f2b472d34344e9ecf786d7692c7cfaa95c"
REPO_DIR = Path("/kaggle/working/Qaci-datascience")


def run_git(*arguments: str) -> None:
    subprocess.run(["git", "-C", str(REPO_DIR), *arguments], check=True)


if not (REPO_DIR / ".git").is_dir():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)],
        check=True,
    )
run_git("fetch", "--depth", "1", "origin", REPO_COMMIT)
run_git("checkout", "--detach", REPO_COMMIT)
GIT_COMMIT = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
).strip()
if GIT_COMMIT != REPO_COMMIT:
    raise RuntimeError(f"Expected {REPO_COMMIT}, checked out {GIT_COMMIT}")
PREREGISTRATION_PATH = (
    REPO_DIR / "docs" / "experiments" / "e03_screening_preregistration.md"
)
PREREGISTRATION_MARKERS = (
    "delta_oof_auc_vs_baseline >= +0.0005",
    "positive_fold_count_vs_baseline >= 4/5",
    "RULE_LOCKED_AFTER_RESULTS = true",
)
if not PREREGISTRATION_PATH.is_file():
    raise RuntimeError(f"Missing pre-registration: {PREREGISTRATION_PATH}")
preregistration_text = PREREGISTRATION_PATH.read_text(encoding="utf-8")
missing_markers = [
    marker for marker in PREREGISTRATION_MARKERS if marker not in preregistration_text
]
if missing_markers:
    raise RuntimeError(f"Pre-registration rule mismatch: {missing_markers}")
SCREENING_MIN_OOF_DELTA = 0.0005
SCREENING_MIN_POSITIVE_FOLDS = 4
SRC_DIR = (REPO_DIR / "src").resolve()
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
importlib.invalidate_caches()

from credit_scoring.artifacts import export_dataframe_artifact, export_json_artifact
from credit_scoring.data.home_credit import audit_home_credit_data, load_home_credit_data
from credit_scoring.evaluation.cross_validation import create_stratified_folds
from credit_scoring.experiments import run_ablation
from credit_scoring.experiments.home_credit_application import prepare_application_data
from credit_scoring.numeric import safe_divide
from credit_scoring.reproducibility import set_global_seed

set_global_seed(42)
print("Git commit:", GIT_COMMIT)
print("Python:", platform.python_version())

In [ ]:
# 2. Pre-registered full screening configuration
RUN_MODES = {
    "smoke": {
        "sample_size": 5_000,
        "n_splits": 3,
        "n_estimators": 300,
        "early_stopping_rounds": 50,
    },
    "screening": {
        "sample_size": None,
        "n_splits": 5,
        "n_estimators": 5_000,
        "early_stopping_rounds": 200,
    },
}
CONFIG = {
    "experiment_name": "E03_bureau_feature_family_ablation",
    "run_mode": "screening",
    "allow_full_screening": True,
    "data_dir": "/kaggle/input/competitions/home-credit-default-risk",
    "output_dir": "/kaggle/working/home_credit_outputs",
    "random_state": 42,
}
if CONFIG["run_mode"] not in RUN_MODES:
    raise ValueError(f"Unknown run mode: {CONFIG['run_mode']}")
if CONFIG["run_mode"] == "screening" and not CONFIG["allow_full_screening"]:
    raise RuntimeError("Full screening requires explicit Checkpoint 3 approval.")
MODE = RUN_MODES[CONFIG["run_mode"]]
MODEL_CONFIG = {
    "learning_rate": 0.02,
    "n_estimators": MODE["n_estimators"],
    "num_leaves": 31,
    "max_depth": -1,
    "min_child_samples": 80,
    "subsample": 0.8,
    "colsample_bytree": 0.7,
    "reg_alpha": 0.1,
    "reg_lambda": 5.0,
    "random_state": CONFIG["random_state"],
    "n_jobs": -1,
    "verbosity": -1,
}
VALIDATION_CONFIG = {
    "n_splits": MODE["n_splits"],
    "shuffle": True,
    "random_state": CONFIG["random_state"],
    "early_stopping_rounds": MODE["early_stopping_rounds"],
    "keep_models": False,
}
OUTPUT_DIR = (
    Path(CONFIG["output_dir"])
    / CONFIG["experiment_name"]
    / CONFIG["run_mode"]
    / GIT_COMMIT[:8]
).resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Run mode:", CONFIG["run_mode"])
print("Output directory:", OUTPUT_DIR)

In [ ]:
# 3. Dataset-specific feature contract and aggregation functions
FAMILY_ORDER = ("counts", "amounts", "recency", "delinquency")
FAMILY_FEATURES = {
    "counts": (
        "BUREAU_LOAN_COUNT",
        "BUREAU_ACTIVE_COUNT",
        "BUREAU_SOLD_COUNT",
        "BUREAU_CREDIT_TYPE_NUNIQUE",
        "BUREAU_CREDIT_PROLONG_SUM",
        "BUREAU_ACTIVE_LOAN_RATIO",
        "BUREAU_BB_COVERAGE_RATIO",
    ),
    "amounts": (
        "BUREAU_CREDIT_SUM",
        "BUREAU_CREDIT_MAX",
        "BUREAU_DEBT_SUM",
        "BUREAU_DEBT_MEAN",
        "BUREAU_DEBT_MAX",
        "BUREAU_ACTIVE_CREDIT_SUM",
        "BUREAU_ACTIVE_DEBT_SUM",
        "BUREAU_OVERDUE_SUM",
        "BUREAU_LIMIT_SUM",
        "BUREAU_ANNUITY_SUM",
        "BUREAU_DEBT_CREDIT_RATIO",
        "BUREAU_OVERDUE_DEBT_RATIO",
    ),
    "recency": (
        "BUREAU_DAYS_CREDIT_MAX",
        "BUREAU_DAYS_CREDIT_MIN",
        "BUREAU_DAYS_CREDIT_MEAN",
        "BUREAU_ACTIVE_ENDDATE_MAX",
        "BUREAU_CLOSED_ENDDATE_FACT_MAX",
        "BUREAU_CLOSED_ENDDATE_DELAY_MEAN",
        "BUREAU_BB_MONTHS_BALANCE_MIN",
        "BUREAU_BB_MONTHS_BALANCE_MAX",
        "BUREAU_BB_MONTHS_SINCE_LAST_DPD",
    ),
    "delinquency": (
        "BUREAU_CREDIT_DAY_OVERDUE_MAX",
        "BUREAU_BB_MONTH_COUNT",
        "BUREAU_BB_STATUS_0_RATIO_TOTAL",
        "BUREAU_BB_STATUS_C_RATIO_TOTAL",
        "BUREAU_BB_ANY_DPD_RATIO_TOTAL",
        "BUREAU_BB_ANY_DPD_RATIO_OBSERVED",
        "BUREAU_BB_SEVERE_DPD_RATIO_TOTAL",
        "BUREAU_BB_SEVERE_DPD_RATIO_OBSERVED",
    ),
}
COUNT_FEATURES = (
    "BUREAU_LOAN_COUNT",
    "BUREAU_ACTIVE_COUNT",
    "BUREAU_SOLD_COUNT",
    "BUREAU_CREDIT_TYPE_NUNIQUE",
    "BUREAU_CREDIT_PROLONG_SUM",
    "BUREAU_BB_MONTH_COUNT",
)
BB_COUNT_COLUMNS = (
    "BB_MONTH_COUNT",
    "BB_STATUS_0_COUNT",
    "BB_STATUS_C_COUNT",
    "BB_OBSERVED_COUNT",
    "BB_ANY_DPD_COUNT",
    "BB_SEVERE_DPD_COUNT",
    "BB_HAS_HISTORY",
)
EXPERIMENT_FAMILIES = {
    "E03-BASE": (),
    "E03-A_counts": ("counts",),
    "E03-B_amounts": ("amounts",),
    "E03-C_recency": ("recency",),
    "E03-D_delinquency": ("delinquency",),
    "E03-ALL": FAMILY_ORDER,
}
assert sum(len(values) for values in FAMILY_FEATURES.values()) == 36


def _normalise_families(families):
    requested = set(FAMILY_ORDER if families is None else families)
    unknown = sorted(requested.difference(FAMILY_ORDER))
    if unknown:
        raise ValueError(f"Unknown Bureau families: {unknown}")
    return tuple(name for name in FAMILY_ORDER if name in requested)


def _require_columns(frame, columns, label):
    missing = sorted(set(columns).difference(frame.columns))
    if missing:
        raise ValueError(f"{label} is missing columns: {missing}")


def aggregate_bureau_balance(frame):
    """Aggregate monthly history to SK_ID_BUREAU without ordering C or X.

    MONTHS_BALANCE is relative to APPLICATION DATE; -1 is the nearest month.
    ANY_DPD is STATUS 1-5 and SEVERE_DPD is STATUS 3-5. Loan-level months
    since last DPD equals -MAX(MONTHS_BALANCE | ANY_DPD).
    """
    _require_columns(
        frame,
        ["SK_ID_BUREAU", "MONTHS_BALANCE", "STATUS"],
        "bureau_balance",
    )
    if frame.duplicated(["SK_ID_BUREAU", "MONTHS_BALANCE"]).any():
        raise ValueError("bureau_balance composite key is not unique.")
    grouped = frame.groupby("SK_ID_BUREAU", sort=False, observed=True)
    result = grouped.agg(
        BB_MONTH_COUNT=("MONTHS_BALANCE", "size"),
        BB_MONTHS_BALANCE_MIN=("MONTHS_BALANCE", "min"),
        BB_MONTHS_BALANCE_MAX=("MONTHS_BALANCE", "max"),
    )
    status_counts = (
        frame.groupby(
            ["SK_ID_BUREAU", "STATUS"],
            sort=False,
            observed=True,
        )
        .size()
        .unstack(fill_value=0)
        .reindex(
            columns=["0", "1", "2", "3", "4", "5", "C", "X"],
            fill_value=0,
        )
    )
    result["BB_STATUS_0_COUNT"] = status_counts["0"]
    result["BB_STATUS_C_COUNT"] = status_counts["C"]
    result["BB_OBSERVED_COUNT"] = status_counts[
        ["0", "1", "2", "3", "4", "5"]
    ].sum(axis=1)
    result["BB_ANY_DPD_COUNT"] = status_counts[
        ["1", "2", "3", "4", "5"]
    ].sum(axis=1)
    result["BB_SEVERE_DPD_COUNT"] = status_counts[["3", "4", "5"]].sum(axis=1)
    last_dpd_month = (
        frame.loc[frame["STATUS"].isin(["1", "2", "3", "4", "5"])]
        .groupby("SK_ID_BUREAU", sort=False, observed=True)["MONTHS_BALANCE"]
        .max()
    )
    result["BB_MONTHS_SINCE_LAST_DPD"] = (-last_dpd_month).astype("float32")
    result["BB_HAS_HISTORY"] = np.int8(1)
    for column in BB_COUNT_COLUMNS:
        result[column] = result[column].astype("int32")
    result = result.reset_index()
    if not result["SK_ID_BUREAU"].is_unique:
        raise AssertionError(
            "bureau_balance aggregation is not unique by SK_ID_BUREAU"
        )
    return result


def aggregate_bureau(bureau, bb_feats, families=None):
    """Aggregate loans to SK_ID_CURR for selected research families.

    DAYS_CREDIT closer to zero is closer to application; MAX(DAYS_CREDIT) is
    the newest loan. Active DAYS_CREDIT_ENDDATE is expected, whereas Closed
    DAYS_ENDDATE_FACT is actual. Their populations are never mixed.
    """
    selected = _normalise_families(families)
    required = [
        "SK_ID_CURR",
        "SK_ID_BUREAU",
        "CREDIT_ACTIVE",
        "CREDIT_TYPE",
        "CNT_CREDIT_PROLONG",
        "AMT_CREDIT_SUM",
        "AMT_CREDIT_SUM_DEBT",
        "AMT_CREDIT_SUM_OVERDUE",
        "AMT_CREDIT_SUM_LIMIT",
        "AMT_ANNUITY",
        "DAYS_CREDIT",
        "DAYS_CREDIT_ENDDATE",
        "DAYS_ENDDATE_FACT",
        "CREDIT_DAY_OVERDUE",
    ]
    _require_columns(bureau, required, "bureau")
    _require_columns(
        bb_feats,
        [
            "SK_ID_BUREAU",
            *BB_COUNT_COLUMNS,
            "BB_MONTHS_BALANCE_MIN",
            "BB_MONTHS_BALANCE_MAX",
            "BB_MONTHS_SINCE_LAST_DPD",
        ],
        "bureau_balance features",
    )
    if (
        not bureau["SK_ID_BUREAU"].is_unique
        or not bb_feats["SK_ID_BUREAU"].is_unique
    ):
        raise AssertionError(
            "Bureau inputs must be unique by SK_ID_BUREAU before merge"
        )
    merged = bureau.merge(
        bb_feats,
        on="SK_ID_BUREAU",
        how="left",
        validate="one_to_one",
    )
    if len(merged) != len(bureau):
        raise AssertionError("Bureau/BB merge changed row count")
    for column in BB_COUNT_COLUMNS:
        merged[column] = merged[column].fillna(0).astype("int32")
    grouped = merged.groupby("SK_ID_CURR", sort=False, observed=True)
    result = pd.DataFrame(index=grouped.size().index)

    if "counts" in selected:
        result["BUREAU_LOAN_COUNT"] = grouped.size().astype("int32")
        result["BUREAU_ACTIVE_COUNT"] = grouped["CREDIT_ACTIVE"].apply(
            lambda values: values.eq("Active").sum()
        ).astype("int32")
        result["BUREAU_SOLD_COUNT"] = grouped["CREDIT_ACTIVE"].apply(
            lambda values: values.eq("Sold").sum()
        ).astype("int32")
        result["BUREAU_CREDIT_TYPE_NUNIQUE"] = grouped[
            "CREDIT_TYPE"
        ].nunique(dropna=True).astype("int32")
        result["BUREAU_CREDIT_PROLONG_SUM"] = (
            grouped["CNT_CREDIT_PROLONG"]
            .sum(min_count=1)
            .fillna(0)
            .astype("int32")
        )
        result["BUREAU_ACTIVE_LOAN_RATIO"] = safe_divide(
            result["BUREAU_ACTIVE_COUNT"],
            result["BUREAU_LOAN_COUNT"],
        )
        covered = grouped["BB_HAS_HISTORY"].sum(min_count=1)
        result["BUREAU_BB_COVERAGE_RATIO"] = safe_divide(
            covered,
            result["BUREAU_LOAN_COUNT"],
        )

    if "amounts" in selected:
        result["BUREAU_CREDIT_SUM"] = grouped["AMT_CREDIT_SUM"].sum(min_count=1)
        result["BUREAU_CREDIT_MAX"] = grouped["AMT_CREDIT_SUM"].max()
        result["BUREAU_DEBT_SUM"] = grouped["AMT_CREDIT_SUM_DEBT"].sum(
            min_count=1
        )
        result["BUREAU_DEBT_MEAN"] = grouped["AMT_CREDIT_SUM_DEBT"].mean()
        result["BUREAU_DEBT_MAX"] = grouped["AMT_CREDIT_SUM_DEBT"].max()
        active = merged.loc[
            merged["CREDIT_ACTIVE"].eq("Active")
        ].groupby("SK_ID_CURR", sort=False, observed=True)
        result["BUREAU_ACTIVE_CREDIT_SUM"] = active["AMT_CREDIT_SUM"].sum(
            min_count=1
        )
        result["BUREAU_ACTIVE_DEBT_SUM"] = active[
            "AMT_CREDIT_SUM_DEBT"
        ].sum(min_count=1)
        result["BUREAU_OVERDUE_SUM"] = grouped[
            "AMT_CREDIT_SUM_OVERDUE"
        ].sum(min_count=1)
        result["BUREAU_LIMIT_SUM"] = grouped["AMT_CREDIT_SUM_LIMIT"].sum(
            min_count=1
        )
        result["BUREAU_ANNUITY_SUM"] = grouped["AMT_ANNUITY"].sum(min_count=1)
        result["BUREAU_DEBT_CREDIT_RATIO"] = safe_divide(
            result["BUREAU_DEBT_SUM"],
            result["BUREAU_CREDIT_SUM"],
        )
        result["BUREAU_OVERDUE_DEBT_RATIO"] = safe_divide(
            result["BUREAU_OVERDUE_SUM"],
            result["BUREAU_DEBT_SUM"],
        )

    if "recency" in selected:
        result["BUREAU_DAYS_CREDIT_MAX"] = grouped["DAYS_CREDIT"].max()
        result["BUREAU_DAYS_CREDIT_MIN"] = grouped["DAYS_CREDIT"].min()
        result["BUREAU_DAYS_CREDIT_MEAN"] = grouped["DAYS_CREDIT"].mean()
        active = merged.loc[
            merged["CREDIT_ACTIVE"].eq("Active")
        ].groupby("SK_ID_CURR", sort=False, observed=True)
        closed_rows = merged.loc[merged["CREDIT_ACTIVE"].eq("Closed")].copy()
        closed_rows["_CLOSED_ENDDATE_DELAY"] = (
            closed_rows["DAYS_ENDDATE_FACT"]
            - closed_rows["DAYS_CREDIT_ENDDATE"]
        )
        closed = closed_rows.groupby("SK_ID_CURR", sort=False, observed=True)
        result["BUREAU_ACTIVE_ENDDATE_MAX"] = active[
            "DAYS_CREDIT_ENDDATE"
        ].max()
        result["BUREAU_CLOSED_ENDDATE_FACT_MAX"] = closed[
            "DAYS_ENDDATE_FACT"
        ].max()
        result["BUREAU_CLOSED_ENDDATE_DELAY_MEAN"] = closed[
            "_CLOSED_ENDDATE_DELAY"
        ].mean()
        result["BUREAU_BB_MONTHS_BALANCE_MIN"] = grouped[
            "BB_MONTHS_BALANCE_MIN"
        ].min()
        result["BUREAU_BB_MONTHS_BALANCE_MAX"] = grouped[
            "BB_MONTHS_BALANCE_MAX"
        ].max()
        result["BUREAU_BB_MONTHS_SINCE_LAST_DPD"] = grouped[
            "BB_MONTHS_SINCE_LAST_DPD"
        ].min()

    if "delinquency" in selected:
        result["BUREAU_CREDIT_DAY_OVERDUE_MAX"] = grouped[
            "CREDIT_DAY_OVERDUE"
        ].max()
        result["BUREAU_BB_MONTH_COUNT"] = (
            grouped["BB_MONTH_COUNT"]
            .sum(min_count=1)
            .fillna(0)
            .astype("int32")
        )
        status_0 = grouped["BB_STATUS_0_COUNT"].sum(min_count=1)
        status_c = grouped["BB_STATUS_C_COUNT"].sum(min_count=1)
        observed = grouped["BB_OBSERVED_COUNT"].sum(min_count=1)
        any_dpd = grouped["BB_ANY_DPD_COUNT"].sum(min_count=1)
        severe_dpd = grouped["BB_SEVERE_DPD_COUNT"].sum(min_count=1)
        total = result["BUREAU_BB_MONTH_COUNT"]
        result["BUREAU_BB_STATUS_0_RATIO_TOTAL"] = safe_divide(status_0, total)
        result["BUREAU_BB_STATUS_C_RATIO_TOTAL"] = safe_divide(status_c, total)
        result["BUREAU_BB_ANY_DPD_RATIO_TOTAL"] = safe_divide(any_dpd, total)
        result["BUREAU_BB_ANY_DPD_RATIO_OBSERVED"] = safe_divide(
            any_dpd,
            observed,
        )
        result["BUREAU_BB_SEVERE_DPD_RATIO_TOTAL"] = safe_divide(
            severe_dpd,
            total,
        )
        result["BUREAU_BB_SEVERE_DPD_RATIO_OBSERVED"] = safe_divide(
            severe_dpd,
            observed,
        )

    selected_columns = [
        column for family in selected for column in FAMILY_FEATURES[family]
    ]
    result = result.reset_index()[["SK_ID_CURR", *selected_columns]]
    for column in selected_columns:
        if column in COUNT_FEATURES:
            result[column] = result[column].astype("int32")
        else:
            result[column] = pd.to_numeric(
                result[column], errors="coerce"
            ).astype("float32")
    numeric = result.select_dtypes(include="number")
    if np.isinf(numeric.to_numpy(dtype=float, na_value=np.nan)).any():
        raise ValueError("Bureau features contain infinite values")
    if not result["SK_ID_CURR"].is_unique:
        raise AssertionError("Bureau aggregation is not unique by SK_ID_CURR")
    return result


def build_bureau_features(bureau, bureau_balance, *, families=None):
    """Build the two-stage Home Credit Bureau research feature matrix."""
    started = time.perf_counter()
    bb_started = time.perf_counter()
    bb_features = aggregate_bureau_balance(bureau_balance)
    bb_runtime = time.perf_counter() - bb_started
    bureau_started = time.perf_counter()
    bureau_features = aggregate_bureau(
        bureau,
        bb_features,
        families=families,
    )
    bureau_runtime = time.perf_counter() - bureau_started
    bureau_features.attrs["aggregation_diagnostics"] = {
        "bureau_rows": len(bureau),
        "bureau_balance_rows": len(bureau_balance),
        "bureau_unique_sk_id_bureau": int(bureau["SK_ID_BUREAU"].nunique()),
        "bureau_balance_unique_sk_id_bureau": int(
            bureau_balance["SK_ID_BUREAU"].nunique()
        ),
        "bureau_unique_sk_id_curr": int(bureau["SK_ID_CURR"].nunique()),
        "bb_feature_rows": len(bb_features),
        "bureau_feature_rows": len(bureau_features),
        "bureau_feature_count": int(bureau_features.shape[1] - 1),
        "bureau_balance_covered_loan_count": int(
            bureau["SK_ID_BUREAU"].isin(bb_features["SK_ID_BUREAU"]).sum()
        ),
        "bureau_balance_loan_coverage_ratio": float(
            bureau["SK_ID_BUREAU"].isin(bb_features["SK_ID_BUREAU"]).mean()
        ),
        "bb_runtime_seconds": bb_runtime,
        "bureau_runtime_seconds": bureau_runtime,
        "total_aggregation_runtime_seconds": time.perf_counter() - started,
    }
    return bureau_features


def merge_bureau_features(application, bureau_features):
    """Left join by applicant; fill counts only, never all missing values."""
    if not application["SK_ID_CURR"].is_unique:
        raise AssertionError("Application SK_ID_CURR must be unique")
    if not bureau_features["SK_ID_CURR"].is_unique:
        raise AssertionError("Bureau features SK_ID_CURR must be unique")
    merged = application.merge(
        bureau_features,
        on="SK_ID_CURR",
        how="left",
        validate="one_to_one",
        sort=False,
    )
    if len(merged) != len(application):
        raise AssertionError("Application/Bureau merge changed row count")
    if not np.array_equal(
        merged["SK_ID_CURR"].to_numpy(),
        application["SK_ID_CURR"].to_numpy(),
    ):
        raise AssertionError("Application/Bureau merge changed row order")
    for column in COUNT_FEATURES:
        if column in merged:
            merged[column] = merged[column].fillna(0).astype("int32")
    return merged

In [ ]:
# 4. Mandatory synthetic assertions — before any real-data read
status_dtype = pd.CategoricalDtype(
    categories=["0", "1", "2", "3", "4", "5", "C", "X"]
)
bureau_id_dtype = pd.CategoricalDtype(categories=[101, 102, 999])
synthetic_bb = pd.DataFrame(
    {
        "SK_ID_BUREAU": pd.Series([101, 101, 102], dtype=bureau_id_dtype),
        "MONTHS_BALANCE": pd.Series([-1, -2, -1], dtype="int16"),
        "STATUS": pd.Series(["0", "1", "C"], dtype=status_dtype),
    }
)
synthetic_bureau = pd.DataFrame(
    {
        "SK_ID_CURR": [1, 1],
        "SK_ID_BUREAU": pd.Series([101, 102], dtype=bureau_id_dtype),
        "CREDIT_ACTIVE": pd.Series(["Active", "Closed"], dtype="category"),
        "CREDIT_TYPE": pd.Series(
            ["Consumer credit", "Credit card"], dtype="category"
        ),
        "CNT_CREDIT_PROLONG": [0, 1],
        "AMT_CREDIT_SUM": [0.0, 0.0],
        "AMT_CREDIT_SUM_DEBT": [10.0, -1.0],
        "AMT_CREDIT_SUM_OVERDUE": [2.0, 0.0],
        "AMT_CREDIT_SUM_LIMIT": [-3.0, np.nan],
        "AMT_ANNUITY": [np.nan, np.nan],
        "DAYS_CREDIT": [-10.0, -200.0],
        "DAYS_CREDIT_ENDDATE": [30.0, -120.0],
        "DAYS_ENDDATE_FACT": [np.nan, -100.0],
        "CREDIT_DAY_OVERDUE": [5.0, 0.0],
    }
)
synthetic_application = pd.DataFrame({"SK_ID_CURR": [1, 2]})
synthetic_bb_features = aggregate_bureau_balance(synthetic_bb)
assert len(synthetic_bb_features) == synthetic_bb["SK_ID_BUREAU"].nunique() == 2
assert synthetic_bb_features["SK_ID_BUREAU"].is_unique
synthetic_direct = aggregate_bureau(
    synthetic_bureau,
    synthetic_bb_features,
    families=FAMILY_ORDER,
)
synthetic_built = build_bureau_features(
    synthetic_bureau,
    synthetic_bb,
    families=FAMILY_ORDER,
)
assert synthetic_built.attrs["aggregation_diagnostics"][
    "bureau_balance_loan_coverage_ratio"
] == 1.0
assert_frame_equal(synthetic_direct, synthetic_built, check_like=False)
assert pd.isna(synthetic_built.loc[0, "BUREAU_ANNUITY_SUM"])
assert pd.isna(synthetic_built.loc[0, "BUREAU_DEBT_CREDIT_RATIO"])
assert synthetic_built.loc[0, "BUREAU_LIMIT_SUM"] == -3.0
assert synthetic_built.loc[0, "BUREAU_BB_MONTHS_SINCE_LAST_DPD"] == 2.0
assert synthetic_built.loc[0, "BUREAU_BB_ANY_DPD_RATIO_TOTAL"] == np.float32(
    1 / 3
)
assert synthetic_built.loc[
    0, "BUREAU_BB_ANY_DPD_RATIO_OBSERVED"
] == np.float32(1 / 2)
assert pd.isna(safe_divide(pd.Series([1.0]), pd.Series([0.0])).iloc[0])
synthetic_merged = merge_bureau_features(
    synthetic_application,
    synthetic_built,
)
no_history = synthetic_merged.loc[
    synthetic_merged["SK_ID_CURR"].eq(2)
].iloc[0]
for column in COUNT_FEATURES:
    assert no_history[column] == 0, f"{column} must be 0 without Bureau history"
for column in [
    "BUREAU_CREDIT_SUM",
    "BUREAU_DAYS_CREDIT_MAX",
    "BUREAU_ACTIVE_LOAN_RATIO",
]:
    assert pd.isna(no_history[column]), f"{column} must stay NaN without history"
print("Synthetic assertions: PASS")

In [ ]:
# 5. Load application first, then sample Bureau coherently by identifiers
DATA_DIR = Path(CONFIG["data_dir"])
application_data = load_home_credit_data(
    DATA_DIR,
    tables=("application_train", "application_test"),
    nrows=MODE["sample_size"],
    reduce_memory=True,
    validate=True,
)
application_train = application_data["application_train"]
application_test = application_data["application_test"]
application_ids = set(application_train["SK_ID_CURR"]).union(
    application_test["SK_ID_CURR"]
)

bureau_header = pd.read_csv(DATA_DIR / "bureau.csv", nrows=0)
category_columns = {"CREDIT_ACTIVE", "CREDIT_CURRENCY", "CREDIT_TYPE"}
bureau_dtype = {}
for column in bureau_header.columns:
    if column in {"SK_ID_CURR", "SK_ID_BUREAU"}:
        bureau_dtype[column] = "int32"
    elif column in category_columns:
        bureau_dtype[column] = "category"
    elif column == "CNT_CREDIT_PROLONG":
        bureau_dtype[column] = "int16"
    else:
        bureau_dtype[column] = "float32"
bureau_all = pd.read_csv(
    DATA_DIR / "bureau.csv",
    dtype=bureau_dtype,
    low_memory=False,
)
bureau_raw_rows = len(bureau_all)
bureau = bureau_all.loc[
    bureau_all["SK_ID_CURR"].isin(application_ids)
].copy()
del bureau_all
gc.collect()
if not bureau["SK_ID_BUREAU"].is_unique:
    raise AssertionError("bureau.SK_ID_BUREAU must be unique")
bureau_ids = set(bureau["SK_ID_BUREAU"])

# Read the complete 27.3M-row table once, never as independent nrows/chunks.
bureau_balance_all = pd.read_csv(
    DATA_DIR / "bureau_balance.csv",
    usecols=["SK_ID_BUREAU", "MONTHS_BALANCE", "STATUS"],
    dtype={
        "SK_ID_BUREAU": "int32",
        "MONTHS_BALANCE": "int16",
        "STATUS": "category",
    },
    low_memory=False,
)
bureau_balance_raw_rows = len(bureau_balance_all)
bureau_balance = bureau_balance_all.loc[
    bureau_balance_all["SK_ID_BUREAU"].isin(bureau_ids)
].copy()
del bureau_balance_all
gc.collect()
if bureau_balance.duplicated(["SK_ID_BUREAU", "MONTHS_BALANCE"]).any():
    raise AssertionError("bureau_balance composite key must be unique")

print("Application train/test rows:", len(application_train), len(application_test))
print("Bureau rows raw/selected:", bureau_raw_rows, len(bureau))
print(
    "Bureau Balance rows raw/selected:",
    bureau_balance_raw_rows,
    len(bureau_balance),
)
display(audit_home_credit_data(application_data))

In [ ]:
# 6. Aggregate once and prepare aligned E01 matrices
all_bureau_features = build_bureau_features(
    bureau,
    bureau_balance,
    families=FAMILY_ORDER,
)
aggregation_diagnostics = dict(
    all_bureau_features.attrs["aggregation_diagnostics"]
)
aggregation_diagnostics.update(
    {
        "bureau_rows_before_id_filter": bureau_raw_rows,
        "bureau_balance_rows_before_id_filter": bureau_balance_raw_rows,
        "application_train_rows": len(application_train),
        "application_test_rows": len(application_test),
    }
)
assert all_bureau_features["SK_ID_CURR"].is_unique
assert all_bureau_features.shape[1] - 1 == 36

prepared_configs = {}
feature_manifests = {}
for experiment_name, families in EXPERIMENT_FAMILIES.items():
    if not families:
        train_input = application_train
        test_input = application_test
        bureau_columns = []
    else:
        bureau_columns = [
            column for family in families for column in FAMILY_FEATURES[family]
        ]
        selected_features = all_bureau_features[
            ["SK_ID_CURR", *bureau_columns]
        ]
        train_input = merge_bureau_features(
            application_train,
            selected_features,
        )
        test_input = merge_bureau_features(
            application_test,
            selected_features,
        )
    prepared = prepare_application_data(
        train_input,
        test_input,
        feature_set="e01",
    )
    prepared_configs[experiment_name] = prepared
    family_by_feature = {
        feature: family
        for family in families
        for feature in FAMILY_FEATURES[family]
    }
    feature_manifests[experiment_name] = pd.DataFrame(
        {
            "feature": prepared.train_features.columns,
            "dtype": [str(dtype) for dtype in prepared.train_features.dtypes],
            "source_family": [
                family_by_feature.get(column, "E01_application")
                for column in prepared.train_features
            ],
            "is_bureau_feature": [
                column in family_by_feature
                for column in prepared.train_features
            ],
        }
    )
    assert list(prepared.train_features.columns) == list(
        prepared.test_features.columns
    )
    assert len(bureau_columns) == sum(
        len(FAMILY_FEATURES[family]) for family in families
    )
    print(
        experiment_name,
        "families=",
        families or ("E01 only",),
        "features=",
        prepared.train_features.shape[1],
    )

del all_bureau_features
gc.collect()

In [ ]:
# 7. Model execution on one immutable fold list
target = prepared_configs["E03-BASE"].target
folds = create_stratified_folds(
    target,
    n_splits=VALIDATION_CONFIG["n_splits"],
    shuffle=VALIDATION_CONFIG["shuffle"],
    random_state=VALIDATION_CONFIG["random_state"],
)
ablation = run_ablation(
    prepared_configs,
    folds,
    baseline_name="E03-BASE",
    model_config=MODEL_CONFIG,
    validation_config=VALIDATION_CONFIG,
)
assert all(
    result["metadata"]["fold_fingerprint"] == ablation["fold_fingerprint"]
    for result in ablation["results"].values()
)
assert all(
    np.all(result["validation_counts"] == 1)
    for result in ablation["results"].values()
)
summary = ablation["summary"]
summary["passes_oof_delta_gate"] = (
    summary["delta_oof_auc_vs_baseline"] >= SCREENING_MIN_OOF_DELTA
)
summary["passes_positive_fold_gate"] = (
    summary["positive_fold_count_vs_baseline"]
    >= SCREENING_MIN_POSITIVE_FOLDS
)
summary["screening_decision"] = np.where(
    summary["experiment"].eq("E03-BASE"),
    "CONTROL",
    np.where(
        summary["passes_oof_delta_gate"]
        & summary["passes_positive_fold_gate"],
        "PASS",
        "FAIL",
    ),
)
if CONFIG["run_mode"] == "smoke":
    print("SMOKE RESULTS — execution evidence only; ignore screening decisions")
else:
    print("FULL SCREENING RESULTS — applying the immutable pre-registered rule")
display(ablation["summary"])

In [ ]:
# 8. Export experiment artifacts as CSV/JSON only
def installed_version(package_name):
    try:
        return version(package_name)
    except PackageNotFoundError:
        return None


artifact_index = {}
for experiment_name, result in ablation["results"].items():
    prepared = prepared_configs[experiment_name]
    experiment_dir = OUTPUT_DIR / "experiments" / experiment_name
    oof_frame = pd.DataFrame(
        {
            "SK_ID_CURR": prepared.train_ids.to_numpy(),
            "TARGET": np.asarray(prepared.target),
            "FOLD": result["fold_assignments"] + 1,
            "OOF_PREDICTION": result["oof_predictions"],
            "VALIDATION_COUNT": result["validation_counts"],
        }
    )
    test_frame = pd.DataFrame(
        {
            "SK_ID_CURR": prepared.test_ids.to_numpy(),
            "TEST_PREDICTION": result["test_predictions"],
        }
    )
    artifact_index[experiment_name] = {
        "oof_predictions": str(
            export_dataframe_artifact(
                oof_frame,
                experiment_dir / "oof_predictions.csv",
            )
        ),
        "test_predictions": str(
            export_dataframe_artifact(
                test_frame,
                experiment_dir / "test_predictions.csv",
            )
        ),
        "feature_importance": str(
            export_dataframe_artifact(
                result["feature_importance"],
                experiment_dir / "feature_importance.csv",
            )
        ),
        "feature_manifest": str(
            export_dataframe_artifact(
                feature_manifests[experiment_name],
                experiment_dir / "feature_manifest.csv",
            )
        ),
    }

summary_path = export_dataframe_artifact(
    ablation["summary"],
    OUTPUT_DIR / "ablation_summary.csv",
)
fold_path = export_dataframe_artifact(
    ablation["fold_metrics"],
    OUTPUT_DIR / "fold_metrics.csv",
)
aggregation_diagnostics["fold_fingerprint"] = ablation["fold_fingerprint"]
aggregation_path = export_json_artifact(
    aggregation_diagnostics,
    OUTPUT_DIR / "aggregation_diagnostics.json",
)
config_path = export_json_artifact(
    {
        "config": CONFIG,
        "mode": MODE,
        "model_config": MODEL_CONFIG,
        "validation_config": VALIDATION_CONFIG,
        "experiments": EXPERIMENT_FAMILIES,
        "feature_families": FAMILY_FEATURES,
    },
    OUTPUT_DIR / "config.json",
)
environment_path = export_json_artifact(
    {
        "python": platform.python_version(),
        "platform": platform.platform(),
        "git_commit": GIT_COMMIT,
        "packages": {
            name: installed_version(name)
            for name in ["numpy", "pandas", "scikit-learn", "lightgbm"]
        },
    },
    OUTPUT_DIR / "environment.json",
)
metadata_path = export_json_artifact(
    {
        "status": f"{CONFIG['run_mode']}_completed",
        "decisive": CONFIG["run_mode"] == "screening",
        "warning": (
            "Smoke metrics are execution evidence only."
            if CONFIG["run_mode"] == "smoke"
            else "Screening decisions use the immutable pre-registered AND rule."
        ),
        "preregistration_path": str(PREREGISTRATION_PATH),
        "screening_rule": {
            "minimum_oof_delta": SCREENING_MIN_OOF_DELTA,
            "minimum_positive_folds": SCREENING_MIN_POSITIVE_FOLDS,
            "total_folds": VALIDATION_CONFIG["n_splits"],
            "operator": "AND",
        },
        "git_commit": GIT_COMMIT,
        "fold_fingerprint": ablation["fold_fingerprint"],
        "oof_coverage_exactly_once": True,
        "artifact_paths": {
            "summary": str(summary_path),
            "fold_metrics": str(fold_path),
            "aggregation_diagnostics": str(aggregation_path),
            "config": str(config_path),
            "environment": str(environment_path),
            "experiments": artifact_index,
        },
    },
    OUTPUT_DIR / "run_metadata.json",
)
print("Experiment artifacts:", OUTPUT_DIR)
print("Run metadata:", metadata_path)

## Screening interpretation

The full-data screening compares every family and E03-ALL with E03-BASE on
the same five folds. A candidate passes only when OOF delta is at least
+0.0005 AND at least 4/5 fold deltas are positive. The rule is locked in the
pre-registration document and must not be changed after results are observed.